# 03 - Forecasting Candidate Selection

## Objective

Identify and select suitable market-level agricultural price series for retrospective forecasting experiments using the cleaned dataset of 46 selected vegetables across India.

This notebook builds upon the time-series analysis performed in Notebook 02.

The objectives are to:

1. Load and examine the preliminary time-series suitability assessment.
2. Evaluate the historical depth and reporting continuity of candidate series.
3. Examine price behaviour and data consistency.
4. Identify representative commodity-market combinations for forecasting.
5. Select and justify a final set of forecasting candidates.
6. Prepare the selected time series for chronological model development.

## Input

- `data/processed/vegetable_prices_cleaned.parquet`
- `data/processed/market_series_suitability.parquet`

## Output

- A documented shortlist of selected market-level forecasting candidates.
- A processed dataset containing the selected historical price series.

## Scope

All 46 selected vegetables are considered during candidate screening. Only series meeting the documented selection requirements will proceed to forecasting experiments.

## Important limitation

The source dataset ends on 20 April 2026. The selected series will therefore be used for retrospective forecasting experiments, not live market-price predictions.

In [1]:
# Import libraries and define project paths

from pathlib import Path

import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
PROCESSED_DIR = ROOT / "data" / "processed"

DATA_FILE = PROCESSED_DIR / "vegetable_prices_cleaned.parquet"
SUITABILITY_FILE = PROCESSED_DIR / "market_series_suitability.parquet"
SUMMARY_FILE = PROCESSED_DIR / "commodity_suitability_summary.csv"

print("Project root:", ROOT)
print("Cleaned dataset exists:", DATA_FILE.exists())
print("Suitability file exists:", SUITABILITY_FILE.exists())
print("Commodity summary exists:", SUMMARY_FILE.exists())

Project root: c:\Users\Lenovo\Documents\GrittingCode\MandiMitra
Cleaned dataset exists: True
Suitability file exists: True
Commodity summary exists: True


In [2]:
# Load and validate preliminary forecasting candidates

suitability = pd.read_parquet(SUITABILITY_FILE)
commodity_suitability = pd.read_csv(SUMMARY_FILE)

candidates = suitability.loc[
    suitability["Criteria_Passed"] == 5
].copy()

print("Total assessed series:", f"{len(suitability):,}")
print("Preliminary candidates:", f"{len(candidates):,}")
print("Unique commodities:", candidates["Project_Commodity"].nunique())
print("Unique states:", candidates["State"].nunique())
print("Unique markets:", candidates["Market"].nunique())

print("\nCandidates by commodity:")
display(
    candidates["Project_Commodity"]
    .value_counts()
    .rename_axis("Commodity")
    .reset_index(name="Candidate_Series")
)

Total assessed series: 70,233
Preliminary candidates: 5,836
Unique commodities: 44
Unique states: 23
Unique markets: 764

Candidates by commodity:


,Commodity,Candidate_Series
0,Brinjal,530
1,Potato,523
2,Onion,498
3,Tomato,456
4,Green Chilli,441
5,Bhindi (Ladies Finger),367
6,Cauliflower,333
7,Bottle gourd,310
8,Pumpkin,296
9,Cabbage,294


In [3]:
# Evaluate historical depth of preliminary candidates

candidates["First_Date"] = pd.to_datetime(candidates["First_Date"])
candidates["Last_Date"] = pd.to_datetime(candidates["Last_Date"])

candidates["History_Years"] = (
    candidates["Last_Date"] - candidates["First_Date"]
).dt.days / 365.25

print("Historical coverage of preliminary candidates")
print("----------------------------------------------")

display(
    candidates["History_Years"]
    .describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90])
    .to_frame("History_Years")
)

print("\nCandidates meeting historical-depth thresholds:")

for years in [2, 3, 5, 8, 10]:
    count = (candidates["History_Years"] >= years).sum()
    print(f"At least {years:2} years: {count:>5,} ({count / len(candidates) * 100:.1f}%)")

Historical coverage of preliminary candidates
----------------------------------------------


,History_Years
count,5836.000000
mean,13.785102
std,5.629869
min,1.702943
10%,6.175222
25%,9.000684
50%,13.813826
75%,18.421629
90%,21.809719
max,24.476386



Candidates meeting historical-depth thresholds:
At least  2 years: 5,801 (99.4%)
At least  3 years: 5,722 (98.0%)
At least  5 years: 5,466 (93.7%)
At least  8 years: 4,772 (81.8%)
At least 10 years: 4,206 (72.1%)


In [4]:
# Apply minimum historical-depth requirement

MIN_HISTORY_YEARS = 5

candidates = candidates.loc[
    candidates["History_Years"] >= MIN_HISTORY_YEARS
].copy()

print("Historical-depth screening")
print("--------------------------")
print(f"Minimum history required: {MIN_HISTORY_YEARS} years")
print(f"Candidates retained: {len(candidates):,}")
print(f"Candidates removed: {5836 - len(candidates):,}")
print(f"Retention rate: {len(candidates) / 5836 * 100:.2f}%")

print("\nCandidates by commodity:")
display(
    candidates["Project_Commodity"]
    .value_counts()
    .rename_axis("Commodity")
    .reset_index(name="Candidate_Series")
)

Historical-depth screening
--------------------------
Minimum history required: 5 years
Candidates retained: 5,466
Candidates removed: 370
Retention rate: 93.66%

Candidates by commodity:


,Commodity,Candidate_Series
0,Brinjal,503
1,Potato,491
2,Onion,478
3,Green Chilli,419
4,Tomato,418
5,Bhindi (Ladies Finger),348
6,Cauliflower,316
7,Bottle gourd,292
8,Pumpkin,277
9,Cabbage,275


In [5]:
# Examine observation density and calendar coverage

candidates["Observations_Per_Year"] = (
    candidates["Observation_Count"] / candidates["History_Years"]
)

print("Observation density")
print("--------------------")

display(
    candidates[
        ["Observation_Count", "Observations_Per_Year", "Calendar_Coverage_Pct"]
    ].describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    ).T
)

print("\nCalendar coverage thresholds:")

for threshold in [10, 20, 30, 40, 50]:
    count = (candidates["Calendar_Coverage_Pct"] >= threshold).sum()
    print(
        f"At least {threshold:2}% coverage: "
        f"{count:>5,} ({count / len(candidates) * 100:.1f}%)"
    )

print("\nObservation density thresholds:")

for threshold in [100, 150, 200, 250, 300]:
    count = (candidates["Observations_Per_Year"] >= threshold).sum()
    print(
        f"At least {threshold:3} observations/year: "
        f"{count:>5,} ({count / len(candidates) * 100:.1f}%)"
    )

Observation density
--------------------


,count,mean,std,min,10%,25%,50%,75%,90%,max
Observation_Count,5466.0,2298.184047,1140.619720,500.000000,938.500000,1408.25000,2137.000000,2982.500000,3896.000000,7181.000000
Observations_Per_Year,5466.0,166.453507,70.336029,23.990364,74.802136,110.44083,164.214262,218.699798,263.726287,360.538697
Calendar_Coverage_Pct,5466.0,45.561857,19.251083,6.570000,20.475000,30.23250,44.950000,59.860000,72.185000,98.680000



Calendar coverage thresholds:
At least 10% coverage: 5,416 (99.1%)
At least 20% coverage: 4,943 (90.4%)
At least 30% coverage: 4,120 (75.4%)
At least 40% coverage: 3,166 (57.9%)
At least 50% coverage: 2,256 (41.3%)

Observation density thresholds:
At least 100 observations/year: 4,357 (79.7%)
At least 150 observations/year: 3,072 (56.2%)
At least 200 observations/year: 1,821 (33.3%)
At least 250 observations/year:   752 (13.8%)
At least 300 observations/year:   154 (2.8%)


In [6]:
# Apply minimum annual observation-density requirement

MIN_OBSERVATIONS_PER_YEAR = 100

before_density_filter = len(candidates)

candidates = candidates.loc[
    candidates["Observations_Per_Year"] >= MIN_OBSERVATIONS_PER_YEAR
].copy()

print("Observation-density screening")
print("------------------------------")
print(f"Minimum observations/year: {MIN_OBSERVATIONS_PER_YEAR}")
print(f"Candidates retained: {len(candidates):,}")
print(f"Candidates removed: {before_density_filter - len(candidates):,}")
print(f"Retention rate: {len(candidates) / before_density_filter * 100:.2f}%")

print("\nCandidates by commodity:")
display(
    candidates["Project_Commodity"]
    .value_counts()
    .rename_axis("Commodity")
    .reset_index(name="Candidate_Series")
)

Observation-density screening
------------------------------
Minimum observations/year: 100
Candidates retained: 4,357
Candidates removed: 1,109
Retention rate: 79.71%

Candidates by commodity:


,Commodity,Candidate_Series
0,Potato,422
1,Brinjal,419
2,Onion,417
3,Green Chilli,360
4,Tomato,354
5,Bottle gourd,252
6,Cauliflower,249
7,Bhindi (Ladies Finger),246
8,Pumpkin,217
9,Cabbage,206


In [7]:
# Examine reporting continuity among retained candidates

print("Reporting-gap distribution")
print("---------------------------")

display(
    candidates[
        [
            "Median_Gap_Days_Gap",
            "P90_Gap_Days_Gap",
            "Maximum_Gap_Days_Gap",
            "Pct_Gaps_Above_30_Days"
        ]
    ].describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    ).T
)

print("\nCandidates meeting P90 reporting-gap thresholds:")

for threshold in [3, 5, 7, 10, 14]:
    count = (
        candidates["P90_Gap_Days_Gap"] <= threshold
    ).sum()

    print(
        f"P90 gap <= {threshold:2} days: "
        f"{count:>5,} ({count / len(candidates) * 100:.1f}%)"
    )

Reporting-gap distribution
---------------------------


,count,mean,std,min,10%,25%,50%,75%,90%,max
Median_Gap_Days_Gap,4357.0,1.033968,0.202698,1.0,1.00,1.0,1.00,1.00,1.00,3.00
P90_Gap_Days_Gap,4357.0,2.627221,1.048037,1.0,2.00,2.0,2.00,3.00,4.00,7.00
Maximum_Gap_Days_Gap,4357.0,560.505164,698.064087,5.0,45.00,121.0,303.00,685.00,1438.80,5461.00
Pct_Gaps_Above_30_Days,4357.0,0.332082,0.307414,0.0,0.04,0.1,0.24,0.48,0.78,2.01



Candidates meeting P90 reporting-gap thresholds:
P90 gap <=  3 days: 3,629 (83.3%)
P90 gap <=  5 days: 4,278 (98.2%)
P90 gap <=  7 days: 4,357 (100.0%)
P90 gap <= 10 days: 4,357 (100.0%)
P90 gap <= 14 days: 4,357 (100.0%)


In [8]:
# Examine the frequency of long reporting interruptions

gap_summary = pd.DataFrame({
    "Threshold": [30, 90, 180, 365, 730],
    "Series_With_Gap": [
        (candidates["Maximum_Gap_Days_Gap"] > days).sum()
        for days in [30, 90, 180, 365, 730]
    ]
})

gap_summary["Percentage"] = (
    gap_summary["Series_With_Gap"] / len(candidates) * 100
)

display(gap_summary)

print("\nExamples with the largest historical gaps:")

display(
    candidates.nlargest(15, "Maximum_Gap_Days_Gap")[
        [
            "Project_Commodity",
            "State",
            "District",
            "Market",
            "First_Date",
            "Last_Date",
            "Observation_Count",
            "Maximum_Gap_Days_Gap",
            "P90_Gap_Days_Gap"
        ]
    ]
)

,Threshold,Series_With_Gap,Percentage
0,30,4057,93.114528
1,90,3499,80.307551
2,180,2832,64.998852
3,365,1909,43.814551
4,730,1027,23.571265



Examples with the largest historical gaps:


,Project_Commodity,State,District,Market,First_Date,Last_Date,Observation_Count,Maximum_Gap_Days_Gap,P90_Gap_Days_Gap
3703,Ginger (Green),Uttar Pradesh,Ballia,Ballia,2002-10-04,2025-11-03,2313,5461,2.0
3591,Potato,Uttar Pradesh,Ballia,Ballia,2002-09-25,2025-10-31,2346,5430,2.0
3110,Onion,Uttar Pradesh,Sitapur,Sitapur,2002-12-11,2025-11-05,2498,5220,2.0
2960,Green Chilli,Uttar Pradesh,Ballia,Ballia,2002-09-25,2025-11-04,2559,5155,2.0
3014,Brinjal,Uttar Pradesh,Ballia,Ballia,2002-09-25,2025-11-01,2538,5155,2.0
4113,Bitter gourd,Odisha,Khurda,Balugaon,2005-07-28,2025-11-04,2185,4821,2.0
2877,Bottle gourd,Uttar Pradesh,Gorakhpur,Gorakhpur,2004-05-24,2025-11-04,2597,4695,2.0
3298,Green Chilli,Chandigarh,Chandigarh,Chandigarh (Grain/Fruit),2002-01-12,2025-11-05,2435,4514,2.0
3480,Potato,Chandigarh,Chandigarh,Chandigarh (Grain/Fruit),2002-01-12,2025-11-05,2382,4403,2.0
3141,Tomato,Chandigarh,Chandigarh,Chandigarh (Grain/Fruit),2002-01-12,2025-11-05,2488,4402,2.0


In [9]:
# Inspect cleaned dataset schema

con = duckdb.connect()

schema = con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{DATA_FILE.as_posix()}')
""").df()

display(schema)

,column_name,column_type,null,key,default,extra
0,State,VARCHAR,YES,None,None,None
1,District,VARCHAR,YES,None,None,None
2,Market,VARCHAR,YES,None,None,None
3,Source_Commodity,VARCHAR,YES,None,None,None
4,Project_Commodity,VARCHAR,YES,None,None,None
5,Variety,VARCHAR,YES,None,None,None
6,Grade,VARCHAR,YES,None,None,None
7,Arrival_Date,VARCHAR,YES,None,None,None
8,Parsed_Date,DATE,YES,None,None,None
9,Min_Price,DOUBLE,YES,None,None,None


In [10]:
# Identify the most recent continuous reporting segment for each candidate

candidate_keys = candidates[
    [
        "State",
        "District",
        "Market",
        "Project_Commodity",
        "Variety",
        "Grade"
    ]
].drop_duplicates()

con.register("candidate_keys", candidate_keys)

recent_segments = con.execute(f"""
    WITH candidate_prices AS (
        SELECT
            p.State,
            p.District,
            p.Market,
            p.Project_Commodity,
            p.Variety,
            p.Grade,
            p.Parsed_Date
        FROM read_parquet('{DATA_FILE.as_posix()}') p
        INNER JOIN candidate_keys c
        USING (
            State, District, Market,
            Project_Commodity, Variety, Grade
        )
        WHERE
            p.Parsed_Date IS NOT NULL
            AND p.is_forecast_eligible = TRUE
            AND p.Modal_Price > 0
            AND isfinite(p.Modal_Price)
    ),

    ordered AS (
        SELECT
            *,
            LAG(Parsed_Date) OVER (
                PARTITION BY
                    State, District, Market,
                    Project_Commodity, Variety, Grade
                ORDER BY Parsed_Date
            ) AS Previous_Date
        FROM candidate_prices
    ),

    marked AS (
        SELECT
            *,
            SUM(
                CASE
                    WHEN Previous_Date IS NULL
                      OR date_diff('day', Previous_Date, Parsed_Date) > 30
                    THEN 1
                    ELSE 0
                END
            ) OVER (
                PARTITION BY
                    State, District, Market,
                    Project_Commodity, Variety, Grade
                ORDER BY Parsed_Date
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) AS Segment_ID
        FROM ordered
    ),

    segment_stats AS (
        SELECT
            State,
            District,
            Market,
            Project_Commodity,
            Variety,
            Grade,
            Segment_ID,
            MIN(Parsed_Date) AS Recent_Segment_Start,
            MAX(Parsed_Date) AS Recent_Segment_End,
            COUNT(*) AS Recent_Segment_Observations
        FROM marked
        GROUP BY
            State, District, Market,
            Project_Commodity, Variety, Grade,
            Segment_ID
    )

    SELECT *
    FROM segment_stats
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY
            State, District, Market,
            Project_Commodity, Variety, Grade
        ORDER BY Segment_ID DESC
    ) = 1

""").df()

# Calculate the duration of each latest segment

recent_segments["Recent_Segment_Years"] = (
    recent_segments["Recent_Segment_End"]
    - recent_segments["Recent_Segment_Start"]
).dt.days / 365.25

recent_segments["Recent_Observations_Per_Year"] = (
    recent_segments["Recent_Segment_Observations"]
    / recent_segments["Recent_Segment_Years"].replace(0, np.nan)
)

print("Most recent continuous reporting segments")
print("-----------------------------------------")
print(f"Candidate series analysed: {len(recent_segments):,}")

display(
    recent_segments[
        [
            "Recent_Segment_Observations",
            "Recent_Segment_Years",
            "Recent_Observations_Per_Year"
        ]
    ].describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    ).T
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Most recent continuous reporting segments
-----------------------------------------
Candidate series analysed: 4,357


,count,mean,std,min,10%,25%,50%,75%,90%,max
Recent_Segment_Observations,4357.0,1125.124857,1015.658116,1.000000,77.000000,240.000000,905.000000,1665.000000,2534.000000,6051.000000
Recent_Segment_Years,4357.0,4.327859,3.802015,0.000000,0.432580,1.062286,3.704312,5.845311,10.490897,23.542779
Recent_Observations_Per_Year,4296.0,250.479143,60.287827,21.072115,175.540676,220.184633,257.532203,288.963685,314.344902,730.500000


In [11]:
# Evaluate the length of the most recent continuous segment

print("Recent continuous-segment thresholds")
print("------------------------------------")

for years in [1, 2, 3, 4, 5]:
    count = (
        recent_segments["Recent_Segment_Years"] >= years
    ).sum()

    print(
        f"At least {years} years: "
        f"{count:>5,} "
        f"({count / len(recent_segments) * 100:.1f}%)"
    )

print("\nRecent segment observation thresholds:")

for observations in [250, 500, 750, 1000, 1500]:
    count = (
        recent_segments["Recent_Segment_Observations"] >= observations
    ).sum()

    print(
        f"At least {observations:4,} observations: "
        f"{count:>5,} "
        f"({count / len(recent_segments) * 100:.1f}%)"
    )

Recent continuous-segment thresholds
------------------------------------
At least 1 years: 3,285 (75.4%)
At least 2 years: 2,828 (64.9%)
At least 3 years: 2,371 (54.4%)
At least 4 years: 2,023 (46.4%)
At least 5 years: 1,628 (37.4%)

Recent segment observation thresholds:
At least  250 observations: 3,251 (74.6%)
At least  500 observations: 2,771 (63.6%)
At least  750 observations: 2,372 (54.4%)
At least 1,000 observations: 2,065 (47.4%)
At least 1,500 observations: 1,263 (29.0%)


In [12]:
# Retain candidates with sufficient recent continuous reporting history

SERIES_KEYS = [
    "State",
    "District",
    "Market",
    "Project_Commodity",
    "Variety",
    "Grade"
]

MIN_RECENT_YEARS = 2
MIN_RECENT_OBSERVATIONS = 500

candidates_with_recent_history = candidates.merge(
    recent_segments,
    on=SERIES_KEYS,
    how="inner"
)

before_filter = len(candidates_with_recent_history)

final_candidates = candidates_with_recent_history.loc[
    (
        candidates_with_recent_history["Recent_Segment_Years"]
        >= MIN_RECENT_YEARS
    )
    &
    (
        candidates_with_recent_history["Recent_Segment_Observations"]
        >= MIN_RECENT_OBSERVATIONS
    )
].copy()

print("Recent-history screening")
print("------------------------")
print(f"Minimum recent history: {MIN_RECENT_YEARS} years")
print(f"Minimum recent observations: {MIN_RECENT_OBSERVATIONS}")
print(f"Candidates before filter: {before_filter:,}")
print(f"Candidates retained: {len(final_candidates):,}")
print(f"Candidates removed: {before_filter - len(final_candidates):,}")
print(f"Retention rate: {len(final_candidates) / before_filter * 100:.2f}%")

print("\nCandidates by commodity:")
display(
    final_candidates["Project_Commodity"]
    .value_counts()
    .rename_axis("Commodity")
    .reset_index(name="Candidate_Series")
)

Recent-history screening
------------------------
Minimum recent history: 2 years
Minimum recent observations: 500
Candidates before filter: 4,357
Candidates retained: 2,714
Candidates removed: 1,643
Retention rate: 62.29%

Candidates by commodity:


,Commodity,Candidate_Series
0,Brinjal,303
1,Onion,302
2,Potato,290
3,Green Chilli,259
4,Tomato,236
5,Pumpkin,146
6,Bottle gourd,131
7,Cauliflower,123
8,Cabbage,118
9,Ginger (Green),106


In [15]:
# Rank forecasting candidates within each commodity

final_candidates = final_candidates.sort_values(
    by=[
        "Project_Commodity",
        "Recent_Segment_Years",
        "Recent_Segment_Observations",
        "Recent_Segment_End"
    ],
    ascending=[True, False, False, False]
).copy()

final_candidates["Candidate_Rank"] = (
    final_candidates
    .groupby("Project_Commodity")
    .cumcount() + 1
)

print("Candidate ranking completed")
print("Total candidates:", f"{len(final_candidates):,}")
print("Commodities:", final_candidates["Project_Commodity"].nunique())

display(
    final_candidates[
        [
            "Project_Commodity",
            "Candidate_Rank",
            "State",
            "District",
            "Market",
            "Variety",
            "Grade",
            "Recent_Segment_Years",
            "Recent_Segment_Observations"
        ]
    ].head(20)
)

Candidate ranking completed
Total candidates: 2,714
Commodities: 39


,Project_Commodity,Candidate_Rank,State,District,Market,Variety,Grade,Recent_Segment_Years,Recent_Segment_Observations
455,Amaranthus,1,Kerala,Kozhikode (Calicut),Palayam,Other,FAQ,15.479808,3934
938,Amaranthus,2,Kerala,Pathanamthitta,Parakkodu,Other,FAQ,15.003422,3028
383,Amaranthus,3,Kerala,Malappuram,Manjeri,Other,FAQ,14.494182,4039
927,Amaranthus,4,Kerala,Alappuzha,Madhavapuram,Other,FAQ,14.234086,3358
602,Amaranthus,5,Kerala,Malappuram,Kondotty,Other,FAQ,14.006845,3779
1345,Amaranthus,6,Kerala,Ernakulam,Angamaly,Amaranthus,FAQ,14.004107,2860
483,Amaranthus,7,Kerala,Alappuzha,Kayamkulam,Other,FAQ,12.793977,3989
923,Amaranthus,8,Kerala,Alappuzha,Harippad,Other,FAQ,12.750171,3373
1268,Amaranthus,9,Kerala,Kollam,Anchal,Other,FAQ,12.153320,3032
1026,Amaranthus,10,Kerala,Ernakulam,Perumbavoor,Amaranthus,FAQ,12.147844,2967


In [17]:
import pandas as pd

# Create a composite key for unique market identification across columns
final_candidates["Market_Identity_Key"] = (
    final_candidates["State"].astype(str)
    + "_"
    + final_candidates["District"].astype(str)
    + "_"
    + final_candidates["Market"].astype(str)
)

# Assess market and state diversity before final candidate selection
diversity_summary = (
    final_candidates.groupby("Project_Commodity")
    .agg(
        Candidate_Series=("Market", "size"),
        Unique_States=("State", "nunique"),
        Unique_Districts=("District", "nunique"),
        Unique_Markets=("Market_Identity_Key", "nunique"),
    )
    .sort_values("Candidate_Series", ascending=False)
    .reset_index()
)

display(diversity_summary)

print("\nOverall diversity")
print("-----------------")
print("Candidate series:", f"{len(final_candidates):,}")
print("Unique commodities:", final_candidates["Project_Commodity"].nunique())
print("Unique states:", final_candidates["State"].nunique())
print(
    "Unique market identities:",
    final_candidates[["State", "District", "Market"]].drop_duplicates().shape[0],
)

,Project_Commodity,Candidate_Series,Unique_States,Unique_Districts,Unique_Markets
0,Brinjal,303,14,160,302
1,Onion,302,15,164,291
2,Potato,290,17,162,289
3,Green Chilli,259,17,146,259
4,Tomato,236,16,148,236
5,Pumpkin,146,14,86,146
6,Bottle gourd,131,10,87,131
7,Cauliflower,123,12,84,123
8,Cabbage,118,14,74,118
9,Ginger (Green),106,11,74,106



Overall diversity
-----------------
Candidate series: 2,714
Unique commodities: 39
Unique states: 18
Unique market identities: 498


In [18]:
# Rank distinct market identities within each commodity

market_keys = [
    "Project_Commodity",
    "State",
    "District",
    "Market"
]

# Keep the strongest variety-grade series for each market
market_candidates = (
    final_candidates
    .sort_values(
        by=[
            "Recent_Segment_Years",
            "Recent_Segment_Observations",
            "Recent_Segment_End"
        ],
        ascending=[False, False, False]
    )
    .drop_duplicates(subset=market_keys, keep="first")
    .copy()
)

# Rank markets within each commodity
market_candidates = market_candidates.sort_values(
    by=[
        "Project_Commodity",
        "Recent_Segment_Years",
        "Recent_Segment_Observations",
        "Recent_Segment_End"
    ],
    ascending=[True, False, False, False]
).copy()

market_candidates["Market_Rank"] = (
    market_candidates
    .groupby("Project_Commodity")
    .cumcount() + 1
)

print("Distinct market candidates:", f"{len(market_candidates):,}")
print("Commodities:", market_candidates["Project_Commodity"].nunique())

display(
    market_candidates[
        [
            "Project_Commodity",
            "Market_Rank",
            "State",
            "District",
            "Market",
            "Variety",
            "Grade",
            "Recent_Segment_Years",
            "Recent_Segment_Observations"
        ]
    ].head(25)
)

Distinct market candidates: 2,699
Commodities: 39


,Project_Commodity,Market_Rank,State,District,Market,Variety,Grade,Recent_Segment_Years,Recent_Segment_Observations
455,Amaranthus,1,Kerala,Kozhikode (Calicut),Palayam,Other,FAQ,15.479808,3934
938,Amaranthus,2,Kerala,Pathanamthitta,Parakkodu,Other,FAQ,15.003422,3028
383,Amaranthus,3,Kerala,Malappuram,Manjeri,Other,FAQ,14.494182,4039
927,Amaranthus,4,Kerala,Alappuzha,Madhavapuram,Other,FAQ,14.234086,3358
602,Amaranthus,5,Kerala,Malappuram,Kondotty,Other,FAQ,14.006845,3779
1345,Amaranthus,6,Kerala,Ernakulam,Angamaly,Amaranthus,FAQ,14.004107,2860
483,Amaranthus,7,Kerala,Alappuzha,Kayamkulam,Other,FAQ,12.793977,3989
923,Amaranthus,8,Kerala,Alappuzha,Harippad,Other,FAQ,12.750171,3373
1268,Amaranthus,9,Kerala,Kollam,Anchal,Other,FAQ,12.153320,3032
1026,Amaranthus,10,Kerala,Ernakulam,Perumbavoor,Amaranthus,FAQ,12.147844,2967


In [19]:
# Select up to two representative markets per commodity

selected_markets = []

for commodity, group in market_candidates.groupby("Project_Commodity"):

    group = group.sort_values("Market_Rank")

    # Always select the highest-ranked market
    first_market = group.iloc[0]
    selected_markets.append(first_market)

    # Prefer a second market from a different state
    remaining = group.iloc[1:]

    different_state = remaining[
        remaining["State"] != first_market["State"]
    ]

    if not different_state.empty:
        second_market = different_state.iloc[0]
        selected_markets.append(second_market)

    elif not remaining.empty:
        selected_markets.append(remaining.iloc[0])

selected_markets = pd.DataFrame(selected_markets).reset_index(drop=True)

print("Final market shortlist")
print("----------------------")
print("Selected market series:", len(selected_markets))
print("Commodities represented:", selected_markets["Project_Commodity"].nunique())
print("States represented:", selected_markets["State"].nunique())
print(
    "Distinct market identities:",
    selected_markets[["State", "District", "Market"]]
    .drop_duplicates()
    .shape[0]
)

display(
    selected_markets[
        [
            "Project_Commodity",
            "State",
            "District",
            "Market",
            "Variety",
            "Grade",
            "Recent_Segment_Years",
            "Recent_Segment_Observations"
        ]
    ].sort_values("Project_Commodity")
)

Final market shortlist
----------------------
Selected market series: 74
Commodities represented: 39
States represented: 12
Distinct market identities: 37


,Project_Commodity,State,District,Market,Variety,Grade,Recent_Segment_Years,Recent_Segment_Observations
0,Amaranthus,Kerala,Kozhikode (Calicut),Palayam,Other,FAQ,15.479808,3934
1,Amaranthus,Kerala,Pathanamthitta,Parakkodu,Other,FAQ,15.003422,3028
2,Beans,Nagaland,Dimapur,Dimapur,Other,FAQ,8.640657,2410
3,Beans,Gujarat,Rajkot,Jasdan,Other,FAQ,4.443532,1032
4,Beetroot,Kerala,Kozhikode (Calicut),Palayam,Other,FAQ,15.479808,3907
...,...,...,...,...,...,...,...,...
69,Tinda,Rajasthan,Bikaner,Bikaner (F&V),Other,FAQ,3.299110,771
70,Tomato,Himachal Pradesh,Kullu,Bhuntar,Other,FAQ,23.022587,5659
71,Tomato,Gujarat,Surendranagar,Vadhvan,Tomato,FAQ,20.793977,6051
72,Yam (Ratalu),Kerala,Idukki,Kattappana,Other,FAQ,5.056810,1003


In [23]:
# Extract historical prices for the selected forecasting candidates

selected_keys = selected_markets[
    SERIES_KEYS + [
        "Recent_Segment_Start",
        "Recent_Segment_End"
    ]
].copy()

selected_keys["Recent_Segment_Start"] = pd.to_datetime(
    selected_keys["Recent_Segment_Start"]
)

selected_keys["Recent_Segment_End"] = pd.to_datetime(
    selected_keys["Recent_Segment_End"]
)

# Use a separate DuckDB connection to avoid unfinished results
forecast_con = duckdb.connect()
forecast_con.register("selected_market_keys", selected_keys)

forecast_prices = forecast_con.execute(f"""
    SELECT
        p.State,
        p.District,
        p.Market,
        p.Project_Commodity,
        p.Variety,
        p.Grade,
        p.Parsed_Date,
        p.Modal_Price
    FROM read_parquet('{DATA_FILE.as_posix()}') p
    INNER JOIN selected_market_keys s
    USING (
        State,
        District,
        Market,
        Project_Commodity,
        Variety,
        Grade
    )
    WHERE
        p.Parsed_Date BETWEEN
            s.Recent_Segment_Start AND s.Recent_Segment_End
        AND p.is_forecast_eligible = TRUE
        AND p.Modal_Price > 0
        AND isfinite(p.Modal_Price)
    ORDER BY
        p.Project_Commodity,
        p.State,
        p.District,
        p.Market,
        p.Parsed_Date
""").df()

print("Forecasting price dataset")
print("-------------------------")
print("Selected series:", len(selected_markets))
print("Price observations:", f"{len(forecast_prices):,}")
print("Commodities:", forecast_prices["Project_Commodity"].nunique())
print(
    "Date range:",
    forecast_prices["Parsed_Date"].min(),
    "to",
    forecast_prices["Parsed_Date"].max()
)

print("\nObservations per selected series:")
display(
    forecast_prices.groupby(SERIES_KEYS)
    .size()
    .describe()
    .to_frame("Observations")
)

display(forecast_prices.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Forecasting price dataset
-------------------------
Selected series: 74
Price observations: 238,755
Commodities: 39
Date range: 2002-04-19 00:00:00 to 2025-11-05 00:00:00

Observations per selected series:


,Observations
count,74.000000
mean,3226.418919
std,1512.375144
min,538.000000
25%,2208.750000
50%,3402.500000
75%,3952.000000
max,6051.000000


,State,District,Market,Project_Commodity,Variety,Grade,Parsed_Date,Modal_Price
0,Kerala,Kozhikode (Calicut),Palayam,Amaranthus,Other,FAQ,2010-05-14,1300.0
1,Kerala,Kozhikode (Calicut),Palayam,Amaranthus,Other,FAQ,2010-05-17,1000.0
2,Kerala,Kozhikode (Calicut),Palayam,Amaranthus,Other,FAQ,2010-05-21,1200.0
3,Kerala,Kozhikode (Calicut),Palayam,Amaranthus,Other,FAQ,2010-05-24,1100.0
4,Kerala,Kozhikode (Calicut),Palayam,Amaranthus,Other,FAQ,2010-05-28,1100.0


In [24]:
# Validate the extracted forecasting dataset

# Count observations in each selected series
extracted_summary = (
    forecast_prices
    .groupby(SERIES_KEYS)
    .agg(
        Extracted_Observations=("Modal_Price", "size"),
        First_Extracted_Date=("Parsed_Date", "min"),
        Last_Extracted_Date=("Parsed_Date", "max")
    )
    .reset_index()
)

# Compare extracted counts with the selected segment counts
validation = selected_markets[
    SERIES_KEYS + [
        "Recent_Segment_Observations",
        "Recent_Segment_Start",
        "Recent_Segment_End"
    ]
].merge(
    extracted_summary,
    on=SERIES_KEYS,
    how="left"
)

validation["Observation_Count_Matches"] = (
    validation["Recent_Segment_Observations"]
    == validation["Extracted_Observations"]
)

# Check duplicate dates within each series
duplicate_rows = forecast_prices.duplicated(
    subset=SERIES_KEYS + ["Parsed_Date"]
).sum()

# Check invalid values
invalid_prices = (
    forecast_prices["Modal_Price"].isna()
    | ~np.isfinite(forecast_prices["Modal_Price"])
    | (forecast_prices["Modal_Price"] <= 0)
).sum()

missing_values = forecast_prices[
    SERIES_KEYS + ["Parsed_Date", "Modal_Price"]
].isna().sum().sum()

print("Forecasting dataset validation")
print("------------------------------")
print("Selected series:", len(selected_markets))
print("Extracted series:", len(extracted_summary))
print(
    "Observation counts match:",
    validation["Observation_Count_Matches"].all()
)
print(
    "Series with mismatched counts:",
    (~validation["Observation_Count_Matches"]).sum()
)
print("Duplicate series-date rows:", duplicate_rows)
print("Invalid prices:", invalid_prices)
print("Missing required values:", missing_values)

display(
    validation.loc[
        ~validation["Observation_Count_Matches"],
        SERIES_KEYS + [
            "Recent_Segment_Observations",
            "Extracted_Observations"
        ]
    ]
)

Forecasting dataset validation
------------------------------
Selected series: 74
Extracted series: 74
Observation counts match: True
Series with mismatched counts: 0
Duplicate series-date rows: 0
Invalid prices: 0
Missing required values: 0


,State,District,Market,Project_Commodity,Variety,Grade,Recent_Segment_Observations,Extracted_Observations


In [25]:
# Save forecasting candidate selection outputs

CANDIDATES_FILE = PROCESSED_DIR / "forecasting_candidates.parquet"
FORECAST_PRICES_FILE = PROCESSED_DIR / "forecasting_candidate_prices.parquet"

selected_markets.to_parquet(
    CANDIDATES_FILE,
    index=False
)

forecast_prices.to_parquet(
    FORECAST_PRICES_FILE,
    index=False
)

print("Saved forecasting datasets:")
print("Candidate shortlist:", CANDIDATES_FILE)
print("Historical prices:", FORECAST_PRICES_FILE)

print("\nFile sizes:")
print(
    "Candidate shortlist:",
    f"{CANDIDATES_FILE.stat().st_size / (1024**2):.2f} MB"
)
print(
    "Historical prices:",
    f"{FORECAST_PRICES_FILE.stat().st_size / (1024**2):.2f} MB"
)

Saved forecasting datasets:
Candidate shortlist: c:\Users\Lenovo\Documents\GrittingCode\MandiMitra\data\processed\forecasting_candidates.parquet
Historical prices: c:\Users\Lenovo\Documents\GrittingCode\MandiMitra\data\processed\forecasting_candidate_prices.parquet

File sizes:
Candidate shortlist: 0.04 MB
Historical prices: 0.68 MB


## Final Candidate Selection Summary

The forecasting candidate selection process began with the preliminary suitability assessment from Notebook 02 and applied additional historical-depth and reporting-continuity requirements.

### Selection criteria

A market-level series was retained if it met all of the following requirements:

1. At least 5 years of overall historical coverage.
2. At least 100 observations per year across its historical span.
3. At least 2 years of data in its latest continuous reporting segment.
4. At least 500 observations in that latest continuous segment.

The latest continuous segment was identified by treating reporting gaps exceeding 30 days as segment boundaries.

### Final selection

- **39 commodities** had qualifying market-level candidates.
- **74 market-level series** were selected, with up to two markets per commodity.
- The second market was preferentially selected from a different state where available.
- The selected series covered **12 states**.
- The extracted historical dataset contained **238,755 observations**.
- All extracted series matched their expected observation counts.
- Validation found no duplicate series-date records, missing required values, or invalid prices.

### Output files

| File | Description |
|---|---|
| `forecasting_candidates.parquet` | Selected market-level forecasting series and their metadata |
| `forecasting_candidate_prices.parquet` | Historical prices from the selected latest continuous segments |

### Limitations

The source dataset ends on 20 April 2026. These historical observations support retrospective forecasting experiments and do not represent live market conditions.

The selected candidates are intended for comparative forecasting experiments. Their selection does not establish that every series is equally predictable or that forecasts can guarantee profitable selling decisions.

The next notebook will use chronological train-validation-test splits to evaluate forecasting models without future-data leakage.